# Milestone 3 lab, notebook 7 of 8: A contract for the supplier

## Where we are

| Notebook | What you do |
|---|---|
| 1. A price has a start and an end | products. See one forgotten date count every sale twice |
| 2. Six rows, wrong everywhere | stores. Find out why the weakest check is the best one here |
| 3. A gap is not a zero | stock_counts. Look for a row that is not there |
| 4. The same person twice | members. Find one person behind two keys, and keep their data private |
| 5. A file you do not control | supplier_prices. A supplier renames a column, and nothing fails |
| 6. Same shape, new meaning | supplier_prices. Every number is valid, and 100 times too high |
| **7. A contract for the supplier** (you are here) | Write down what a file must look like, and check it at the door |
| 8. Cards side by side | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
import csv
import tempfile
from decimal import Decimal
from pathlib import Path

import pandas as pd

from lab_helpers import LANDING, prepare_week
from lotus_pipeline import contract, pipeline

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: The phone call

Finance has now had two bad mornings: an order with no cost, and an order 100 times too dear. Someone at Lotus phones the supplier.

Lotus says: "You renamed our column." The supplier says: "Your column? It is our file."

Lotus says: "The cost must be in dollars." The supplier says: "Where does it say that?"

Nowhere. Lotus **assumed** the file would stay as it was. The supplier never promised it.

Both checks you wrote speak after the load, when the bad rows are already in the warehouse. And when they fire, there is nothing to point at and say: this is what we agreed.

## Step 2: Write it down

A **data contract** is a written agreement between the producer of data and its consumers. It has four parts: an owner, the columns, the rules for the values, and a change policy.

Lotus's contract for this file is one short page: `contracts/supplier_prices.yml`. Read it part by part. First, who.

In [2]:
agreed = contract.load_contract()
print("Contract:", agreed["name"], "version", agreed["version"])
for part, text in agreed["owner"].items():
    print(f"  {part:<9} {text}")

Contract: supplier_prices version 1.0.0
  producer  Each supplier that sells to Lotus Retails (SUP-A, SUP-B, SUP-C)
  consumer  Lotus Retails finance, who pay suppliers from these costs
  contact   buying@lotus-retails.example


The **producer** makes the data. The **consumer** reads it. The first use of a contract is simply that both are named, with someone to call.

Second and third: the columns, and the rule for each value.

In [3]:
parts = ["name", "type", "meaning", "allowed", "shape", "decimals", "min", "max"]
pd.DataFrame([{part: column.get(part, "") for part in parts} for column in agreed["columns"]])

,name,type,meaning,allowed,shape,decimals,min,max
0,supplier_id,text,Who sent the file,"[SUP-A, SUP-B, SUP-C]",,,,
1,product_id,text,Lotus's own id of the product,,P-000,,,
2,unit_cost,decimal,"The cost of one piece or one kilogram, in US dollars. Not in cents",,,2,0.01,50.0
3,unit,text,What one unit is,"[each, kg]",,,,
4,price_date,date,The day the cost starts. It is the date in the file name,,,,,


Find the row `unit_cost`. Its meaning is written out: "in US dollars. Not in cents". And its rule says how a dollar amount looks: two decimals, between 0.01 and 50.00.

That one row is the sentence nobody had said on the phone.

Fourth: what may change, and how.

In [4]:
for kind, text in agreed["change_policy"].items():
    print(f"{kind}:")
    print(f"  {text}")
    print()

additive:
  A supplier may add a column at any time. Lotus ignores it and reports it

breaking:
  Renaming or removing a column, or changing its type or its meaning, is a breaking change. It needs 30 days' notice and a new version of this contract



You know both words from notebook 5. An additive change is allowed. A breaking change needs 30 days' notice. Renaming `unit_cost` was a breaking change, made with no notice. With this page, Lotus can say so.

## Step 3: Hold every file against it

A contract that nobody checks is a wish. The production code holds a file against the contract **before** it is loaded. Run it on all fifteen supplier files of the five weeks.

In a terminal, from this milestone's folder: `uv run python -m lotus_pipeline contract`.

In [5]:
_ = pipeline.main(["contract"])

The contract supplier_prices, version 1.0.0

  supplier_prices_SUP-A_2026-07-27.csv       accept
  supplier_prices_SUP-A_2026-08-03.csv       accept
  supplier_prices_SUP-A_2026-08-10.csv       accept
  supplier_prices_SUP-A_2026-08-17.csv       accept
  supplier_prices_SUP-A_2026-08-24.csv       accept
  supplier_prices_SUP-B_2026-07-28.csv       accept
  supplier_prices_SUP-B_2026-08-04.csv       accept
  supplier_prices_SUP-B_2026-08-11.csv       accept
  supplier_prices_SUP-B_2026-08-18.csv       accept
  supplier_prices_SUP-B_2026-08-25.csv       refuse
      breaking  unit_cost: an agreed column is missing
      additive  cost_per_unit: a column that is not in the contract
  supplier_prices_SUP-C_2026-07-31.csv       accept
  supplier_prices_SUP-C_2026-08-07.csv       accept
  supplier_prices_SUP-C_2026-08-14.csv       accept
  supplier_prices_SUP-C_2026-08-21.csv       accept
  supplier_prices_SUP-C_2026-08-28.csv       refuse
      value     unit_cost: 0 decimals, not 2: dollar

Thirteen files keep the contract. Two would be refused at the door, and each says why, in words a supplier can act on.

- `SUP-B`, 25 August: an agreed column is missing. That is the rename from notebook 5.
- `SUP-C`, 28 August: `655` has no decimals. A dollar amount is written like `6.55`. That is the cents from notebook 6.

The same two faults you caught after the load, now caught before it.

## Step 4: How a rule is checked

There is no magic in it. Check the "two decimals" rule yourself, on the file from `SUP-C`, with plain Python.

In [6]:
path = LANDING / "supplier_prices" / "supplier_prices_SUP-C_2026-08-28.csv"
with path.open(newline="") as f:
    rows = list(csv.DictReader(f))

broken = []
for line_number, row in enumerate(rows, start=2):  # line 1 of the file is the header
    decimals = -Decimal(row["unit_cost"]).as_tuple().exponent
    if decimals != 2:
        broken.append((line_number, row["product_id"], row["unit_cost"], decimals))

print(len(rows), "rows in the file.", len(broken), "break the rule 'two decimals'.")
for line_number, product, value, decimals in broken[:3]:
    print(f"  line {line_number}: {product} has unit_cost {value}, with {decimals} decimals")

15 rows in the file. 15 break the rule 'two decimals'.
  line 2: P-024 has unit_cost 655, with 0 decimals
  line 3: P-025 has unit_cost 125, with 0 decimals
  line 4: P-026 has unit_cost 142, with 0 decimals


Read each value, test it against the rule, keep the ones that break it, and say on which line. That is all a contract check is.

The production checker gives the same answer.

In [7]:
problems = contract.check_file(path)
print(len(problems), "problems.", "Verdict:", contract.verdict(problems))
pd.DataFrame(problems).head(3)

15 problems. Verdict: refuse


,kind,column,row,value,problem
0,value,unit_cost,2,655,"0 decimals, not 2: dollars are written like 2.10"
1,value,unit_cost,3,125,"0 decimals, not 2: dollars are written like 2.10"
2,value,unit_cost,4,142,"0 decimals, not 2: dollars are written like 2.10"


## Step 5: Write one more rule

A contract grows. Each time a file surprises you, you add the rule that would have caught it.

Here is a rule the contract does not have yet: **a file names each product once.** If a supplier sent the same product on two lines with two costs, which one would Lotus pay?

In [8]:
def products_named_twice(path: Path) -> list[str]:
    """The products that are on more than one line of a supplier file."""
    with path.open(newline="") as f:
        products = [row["product_id"] for row in csv.DictReader(f)]
    return sorted({p for p in products if products.count(p) > 1})


files = sorted((LANDING / "supplier_prices").glob("*.csv"))
breaking_it = [f.name for f in files if products_named_twice(f)]
print(len(files), "files checked.", len(breaking_it), "name a product twice.")

15 files checked. 0 name a product twice.


All fifteen real files keep the new rule. A rule that is silent on good files is half tested. Now see it fail: write a small file by hand, with one product on two lines.

In [9]:
with tempfile.TemporaryDirectory() as folder:
    bad_file = Path(folder) / "supplier_prices_SUP-A_2026-08-31.csv"
    bad_file.write_text(
        "supplier_id,product_id,unit_cost,unit,price_date\n"
        "SUP-A,P-001,1.76,each,2026-08-31\n"
        "SUP-A,P-002,2.20,each,2026-08-31\n"
        "SUP-A,P-001,1.90,each,2026-08-31\n"
    )
    print("Named twice:", products_named_twice(bad_file))
    print("The contract as it is today:", contract.verdict(contract.check_file(bad_file)))

Named twice: ['P-001']
The contract as it is today: accept


Your rule catches it. The contract as it is written today would accept that file: every value on every line is fine. To make the rule real, you would add it to the page, agree it with the suppliers, and raise the version number.

## Step 6: At the door, or after the load?

| | A check on a table | A contract on a file |
|---|---|---|
| When it speaks | After the load | Before the load, at the door |
| What it knows | Other tables, and history | Only this file, and the page |
| It caught the cents by | Comparing with last week, and with the shelf price | The rule "two decimals" |
| It cannot see | The shape the file had | A cost that is a valid dollar amount and still wrong |
| Who can read it | The data team | Both companies |

You need both. Suppose `SUP-C` sends `65.50` where `6.55` was right: two decimals, but above the largest value of 50.00, so the contract still refuses it. Now suppose they send `19.65`, three times the real cost. That is a perfect dollar amount. Only "at most twice the last file" can catch that one.

Large teams do not check contracts by hand. They use a **schema registry**: a service that stores the agreed schema of every feed, and refuses a breaking change automatically. This project writes the contract by hand, so that you see every part of it. dbt has its own kind of contract for the tables it builds: [dbt: Model Contracts, Versions & Access](https://www.d8loop.com/modules/knowledge-corner/dbt/model-contracts-versions-and-access).

## What you now know

- A **data contract** is a written agreement between the producer of data and its consumers: an owner, the columns, the rules for the values, and a change policy.
- It is checked at the door, before the load. Two of fifteen files would have been refused.
- A rule is plain code: read each value, test it, say which line broke it.
- A contract and a check on the table see different faults. Keep both.

**Say it in an interview:** "After a supplier renamed a column and later sent cents for dollars, I wrote a data contract for that file: who owns it, each column with its type and its rule, and a change policy with 30 days' notice for a breaking change. We check every file against it before loading. It would have refused both bad files at the door, with a reason the supplier could act on."

**The theory:** [Schema Evolution & Data Contracts](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/schema-evolution-and-data-contracts), sections "Write a data contract" and "Change a name safely". The Python side is in [Python: Validation, Contracts & Error Handling](https://www.d8loop.com/modules/knowledge-corner/python/validation-contracts-error-handling).

Next: [notebook 8](08_cards_side_by_side.ipynb). Seven cards are done. Put them side by side.